In [215]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import plotly.express as px


In [216]:
try:
    df=pd.read_csv('Stroke Prediction.csv')
    print("Dataset Loaded Sucessfully")
except:
    print('Dataset not Found')

Dataset Loaded Sucessfully


In [217]:
df

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,72940,Female,2.0,0,0,No,children,Urban,102.92,17.6,Unknown,0
1,72918,Female,53.0,1,0,Yes,Private,Urban,62.55,30.3,Unknown,1
2,72915,Female,45.0,0,0,Yes,Private,Urban,172.33,45.3,formerly smoked,0
3,72914,Female,19.0,0,0,No,Private,Urban,90.57,24.2,Unknown,0
4,72911,Female,57.0,1,0,Yes,Private,Rural,129.54,60.9,smokes,0
...,...,...,...,...,...,...,...,...,...,...,...,...
5105,99,Female,31.0,0,0,No,Private,Urban,108.89,52.3,Unknown,0
5106,91,Female,42.0,0,0,No,Private,Urban,98.53,18.5,never smoked,0
5107,84,Male,55.0,0,0,Yes,Private,Urban,89.17,31.5,never smoked,0
5108,77,Female,13.0,0,0,No,children,Rural,85.81,18.6,Unknown,0


In [218]:
df.isnull().sum().sort_values(ascending=False)

bmi                  201
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
smoking_status         0
stroke                 0
dtype: int64

In [219]:
from sklearn.impute import SimpleImputer
si=SimpleImputer()
df['bmi']=si.fit_transform(df[['bmi']])

In [220]:
df.isnull().sum().sort_values(ascending=False)

id                   0
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64

In [221]:
df=df.drop('id',axis=1)
df

,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Female,2.0,0,0,No,children,Urban,102.92,17.600000,Unknown,0
1,Female,53.0,1,0,Yes,Private,Urban,62.55,30.300000,Unknown,1
2,Female,45.0,0,0,Yes,Private,Urban,172.33,45.300000,formerly smoked,0
3,Female,19.0,0,0,No,Private,Urban,90.57,24.200000,Unknown,0
4,Female,57.0,1,0,Yes,Private,Rural,129.54,60.900000,smokes,0
...,...,...,...,...,...,...,...,...,...,...,...
5105,Female,31.0,0,0,No,Private,Urban,108.89,52.300000,Unknown,0
5106,Female,42.0,0,0,No,Private,Urban,98.53,18.500000,never smoked,0
5107,Male,55.0,0,0,Yes,Private,Urban,89.17,31.500000,never smoked,0
5108,Female,13.0,0,0,No,children,Rural,85.81,18.600000,Unknown,0


In [222]:
X=df.drop('stroke',axis=1)
y=df['stroke']


In [223]:
px.box(df,x="avg_glucose_level")

In [224]:
px.box(df,x="age")

In [225]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector as selector
from sklearn.preprocessing import StandardScaler,LabelEncoder,OneHotEncoder,OrdinalEncoder,RobustScaler

In [226]:
num=Pipeline([
    ('scaler',RobustScaler())
])
cat=Pipeline([
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])
preprocessor=ColumnTransformer([
    ('nums',num,selector(dtype_include=np.number)),
    ('cats',cat,selector(dtype_include=['object','category']))
])

In [227]:
preprocessor

,transformers,"[('nums', ...), ('cats', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"


In [228]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
model=Pipeline([
    ('preprocessor',preprocessor),
    ('model',XGBClassifier())
    
     ])
model

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('nums', ...), ('cats', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [229]:
from sklearn.model_selection import GridSearchCV
param_grid={
    'model__n_estimators':[500],
    'model__max_depth':[5],
    'model__learning_rate':[0.3]
}
cv=GridSearchCV(
    param_grid=param_grid,
    estimator=model,
    verbose=1,
    scoring='r2',
    n_jobs=3


)

In [230]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


In [231]:
cv.fit(X_train, y_train)
y_pred=cv.predict(X_test)


Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [232]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report,f1_score,precision_score,r2_score

In [236]:
print('Accuracy',accuracy_score(y_test,y_pred))
print('f1_score',f1_score(y_test,y_pred))
print('precision_score',precision_score(y_test,y_pred))
print('\nconfusion_matrix\n',confusion_matrix(y_test,y_pred))
print('\nclassification_report\n',classification_report(y_test,y_pred))

Accuracy 0.860078277886497
f1_score 0.13333333333333333
precision_score 0.1

confusion_matrix
 [[868  99]
 [ 44  11]]

classification_report
               precision    recall  f1-score   support

           0       0.95      0.90      0.92       967
           1       0.10      0.20      0.13        55

    accuracy                           0.86      1022
   macro avg       0.53      0.55      0.53      1022
weighted avg       0.91      0.86      0.88      1022

